# Spatial Transcriptomics DE Pipeline — Half Brain
**QC rule:** a gene is kept if detected in ≥ 3 spots in ≥ 3 of 4 samples
within **at least one** experimental group.

**Workflow:**
0. Imports & Setup
1. Read Visium data → individual h5ad
2. Concatenate into unified AnnData
3. Quality Control (violin plots before & after)
4. Normalization
5. Feature Selection (HVGs)
7. PCA-Loading Gene Filtering
8. Clustering & UMAP
9. Pseudobulk Construction
10. Differential Expression (Welch t-test)
11. Volcano Plots
12. Summary

## 0 · Imports & Setup

In [ ]:
import scanpy as sc
import squidpy as sq
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import sparse, stats
from statsmodels.stats.multitest import multipletests
from kneed import KneeLocator
import os, shutil, warnings

try:
    from adjustText import adjust_text
    _HAS_ADJUSTTEXT = True
    print("✓ adjustText loaded — volcano labels will be auto-repelled")
except ImportError:
    _HAS_ADJUSTTEXT = False
    print("⚠ adjustText not installed — using manual offset for volcano labels")
    print("  Install with: pip install adjustText")

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=150, facecolor="white", frameon=False)


In [ ]:

# --- Paths -------------------------------------------------------------------
VISIUM_INPUT_FOLDER          = "/opt/Data/RNA Seq 2025"
H5AD_OUTPUT_PATH             = "/home/ajarrah/PhD_Thesis/gene_paper/h5ad_data/"
AGGREGATED_H5AD_OUTPUT_PATH  = "/home/ajarrah/PhD_Thesis/gene_paper/aggregated_h5ad_data/"
FILE_NAME_LIST_CSV           = "/home/ajarrah/PhD_Thesis/gene_paper/csv_data/file_name_list.csv"
FIGURE_DIR                   = f"figures_halfbrain"
RESULTS_DIR                  = f"results_halfbrain"

SAVE_AGGREGATED = False

for d in [H5AD_OUTPUT_PATH, AGGREGATED_H5AD_OUTPUT_PATH, FIGURE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

sc.settings.figdir = FIGURE_DIR


In [ ]:
# --- Sample definitions (single source of truth) -----------------------------
SAMPLE_DEFS = {
    "A1_Young_Control_Mouse_Brain_202502": ("YC_1",  "YC",  "Control", "Young", "1-1"),
    "B1_Young_Control_Mouse_Brain_202502": ("YC_2",  "YC",  "Control", "Young", "1-2"),
    "C1_Young_Control_Mouse_Brain_202502": ("YC_3",  "YC",  "Control", "Young", "1-3"),
    "D1_Young_Control_Mouse_Brain_202502": ("YC_4",  "YC",  "Control", "Young", "1-4"),
    "A1_Young_AD_Mouse_Brain_202502":      ("YAD_1", "YAD", "AD",      "Young", "2-1"),
    "B1_Young_AD_Mouse_Brain_202502":      ("YAD_2", "YAD", "AD",      "Young", "2-2"),
    "C1_Young_AD_Mouse_Brain_202502":      ("YAD_3", "YAD", "AD",      "Young", "2-3"),
    "D1_Young_AD_Mouse_Brain_202502":      ("YAD_4", "YAD", "AD",      "Young", "2-4"),
    "A1_Aged_Control_Mouse_Brain_202502":  ("AC_1",  "AC",  "Control", "Aged",  "3-1"),
    "B1_Aged_Control_Mouse_Brain_202502":  ("AC_2",  "AC",  "Control", "Aged",  "3-2"),
    "C1_Aged_Control_Mouse_Brain_202502":  ("AC_3",  "AC",  "Control", "Aged",  "3-2"),
    "D1_Aged_Control_Mouse_Brain_202502":  ("AC_4",  "AC",  "Control", "Aged",  "3-4"),
    "A1_Aged_AD_Mouse_Brain_202502":       ("AAD_1", "AAD", "AD",      "Aged",  "4-1"),
    "B1_Aged_AD_Mouse_Brain_202502":       ("AAD_2", "AAD", "AD",      "Aged",  "4-2"),
    "C1_Aged_AD_Mouse_Brain_202502":       ("AAD_3", "AAD", "AD",      "Aged",  "4-3"),
    "D1_Aged_AD_Mouse_Brain_202502":       ("AAD_4", "AAD", "AD",      "Aged",  "4-4"),
}

SAMPLE_ORDER = [
    "YC_1",  "YC_2",  "YC_3",  "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1",  "AC_2",  "AC_3",  "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4",
]

SHORT_TO_ORIG = {v[0]: k for k, v in SAMPLE_DEFS.items()}


In [ ]:
# --- User-defined comparisons ------------------------------------------------
COMPARISONS = [
    {
        "name": "AD_vs_Control",
        "desc": "All AD vs All Control",
        "group1_filter": {"Condition": "AD"},
        "group2_filter": {"Condition": "Control"},
    },
    {
        "name": "Aged_vs_Young",
        "desc": "All Aged vs All Young",
        "group1_filter": {"Age": "Aged"},
        "group2_filter": {"Age": "Young"},
    },
    {
        "name": "YAD_vs_YC",
        "desc": "Young AD vs Young Control",
        "group1_filter": {"Age": "Young", "Condition": "AD"},
        "group2_filter": {"Age": "Young", "Condition": "Control"},
    },
    {
        "name": "AAD_vs_AC",
        "desc": "Aged AD vs Aged Control",
        "group1_filter": {"Age": "Aged", "Condition": "AD"},
        "group2_filter": {"Age": "Aged", "Condition": "Control"},
    },
    {
        "name": "AC_vs_YC",
        "desc": "Aged Control vs Young Control (aging effect)",
        "group1_filter": {"Age": "Aged",  "Condition": "Control"},
        "group2_filter": {"Age": "Young", "Condition": "Control"},
    },
    {
        "name": "AAD_vs_YAD",
        "desc": "Aged AD vs Young AD",
        "group1_filter": {"Age": "Aged",  "Condition": "AD"},
        "group2_filter": {"Age": "Young", "Condition": "AD"},
    },
]


In [ ]:
# --- Helper functions ---------------------------------------------------------

_PALETTE = [
    "#c0392b", "#2980b9", "#8e44ad", "#27ae60", "#d35400",
    "#2c3e50", "#16a085", "#e67e22", "#1abc9c", "#e74c3c",
]
comp_labels = {c["name"]: c["desc"] for c in COMPARISONS}
comp_colors = {c["name"]: _PALETTE[i % len(_PALETTE)]
               for i, c in enumerate(COMPARISONS)}


def build_mask(obs_df, filt):
    mask = pd.Series(True, index=obs_df.index)
    for col, val in filt.items():
        if col not in obs_df.columns:
            raise KeyError(f"Filter column '{col}' not found. Available: {list(obs_df.columns)}")
        if isinstance(val, (list, tuple)):
            mask &= obs_df[col].isin(val)
        else:
            mask &= obs_df[col] == val
    return mask


def describe_filter(filt):
    parts = []
    for col, val in filt.items():
        if isinstance(val, (list, tuple)):
            parts.append(f"{col}∈{{{', '.join(val)}}}")
        else:
            parts.append(f"{col}={val}")
    return " & ".join(parts)


def check_disk_space(path, required_gb=1.0):
    try:
        usage = shutil.disk_usage(os.path.dirname(os.path.abspath(path)))
        free_gb = usage.free / (1024 ** 3)
        print(f"  Disk space available at {path}: {free_gb:.1f} GB")
        if free_gb < required_gb:
            print(f"  ⚠ WARNING: Less than {required_gb} GB free!")
            return False
        return True
    except Exception:
        return True


def safe_write_h5ad(adata_obj, filepath, compression="gzip"):
    check_disk_space(filepath, required_gb=0.5)
    try:
        adata_obj.write(filepath, compression=compression)
        print(f"  ✓ Saved: {filepath}")
    except OSError as e:
        print(f"  ✗ WRITE FAILED: {e}")
        if os.path.exists(filepath):
            os.remove(filepath)
        raise


## 1 · Read Visium Data → Individual h5ad Files

In [ ]:
file_name_list = sorted(os.listdir(VISIUM_INPUT_FOLDER))
pd.DataFrame(file_name_list, columns=["file_name"]).to_csv(FILE_NAME_LIST_CSV, index=False)
print(f"Found {len(file_name_list)} samples in {VISIUM_INPUT_FOLDER}")

for sample_folder in file_name_list:
    if sample_folder not in SAMPLE_DEFS:
        print(f"  ⚠ Skipping unrecognized folder: {sample_folder}")
        continue

    short_key, group, condition, age, code_id = SAMPLE_DEFS[sample_folder]
    out_path = os.path.join(H5AD_OUTPUT_PATH, f"{sample_folder}.h5ad")

    if os.path.exists(out_path):
        print(f"  Skipping {sample_folder} -> {short_key} (already exists)")
        continue

    print(f"  Reading {sample_folder} -> {short_key}")
    a = sc.read_visium(os.path.join(VISIUM_INPUT_FOLDER, sample_folder, "outs"))
    a.obs["Original_Name"] = sample_folder
    a.obs["Sample_Code"]   = code_id
    a.obs["Group"]         = group
    a.var_names_make_unique()
    safe_write_h5ad(a, out_path)
    del a


## 2 · Concatenate into Unified AnnData

In [ ]:
adatas_for_agg = [] if SAVE_AGGREGATED else None
adatas_for_analysis = []

for short_key in SAMPLE_ORDER:
    orig_name = SHORT_TO_ORIG[short_key]
    _, group, condition, age, code_id = SAMPLE_DEFS[orig_name]
    filepath = os.path.join(H5AD_OUTPUT_PATH, f"{orig_name}.h5ad")

    a = sc.read_h5ad(filepath)

    if SAVE_AGGREGATED:
        adatas_for_agg.append(a.copy())

    a.obs["Sample"]    = short_key
    a.obs["Group"]     = group
    a.obs["Condition"] = condition
    a.obs["Age"]       = age
    a.obs_names = [f"{short_key}_{x}" for x in a.obs_names]
    a.obs_names_make_unique()
    adatas_for_analysis.append(a)

if SAVE_AGGREGATED:
    print("Building aggregated spatial object...")
    adata_agg = sc.concat(
        adatas_for_agg, label="Sample", keys=SAMPLE_ORDER, uns_merge="unique"
    )
    for short_key, a_orig in zip(SAMPLE_ORDER, adatas_for_agg):
        if "spatial" not in a_orig.uns:
            continue
        inner_key = next(iter(a_orig.uns["spatial"].keys()))
        if inner_key in adata_agg.uns.get("spatial", {}):
            spatial_dict = adata_agg.uns["spatial"]
            if short_key != inner_key:
                spatial_dict[short_key] = spatial_dict.pop(inner_key)
    agg_path = os.path.join(AGGREGATED_H5AD_OUTPUT_PATH, "aggregated_mouse_brain_202502.h5ad")
    safe_write_h5ad(adata_agg, agg_path)
    print(f"Aggregated object saved: {adata_agg.shape}")
    del adata_agg, adatas_for_agg
else:
    print("Skipping aggregated h5ad save (SAVE_AGGREGATED=False).")

adata = ad.concat(adatas_for_analysis, join="inner", merge="first")
adata.obs_names_make_unique()
del adatas_for_analysis
print(f"Analysis object: {adata.shape}")


## 3 · Quality Control

### 3a — QC Metrics BEFORE Filtering

In [ ]:
# --- Remove all-zero genes ---------------------------------------------------
if sparse.issparse(adata.X):
    mask = np.array(adata.X.sum(axis=0) != 0).ravel()
else:
    mask = adata.X.sum(axis=0) != 0
adata = adata[:, mask].copy()

# --- Mitochondrial & ribosomal flags -----------------------------------------
adata.var["mt"]   = adata.var_names.str.startswith("mt-")
adata.var["ribo"] = adata.var_names.str.match("^(Rpl|Rps)")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True)

print(f"Shape before QC filtering: {adata.shape}")


In [ ]:
# --- Violin plots BEFORE filtering -------------------------------------------
qc_keys = ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"]
qc_titles = ["Genes per spot", "UMI counts per spot", "Mitochondrial %", "Ribosomal %"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()

for ax, key, title in zip(axes, qc_keys, qc_titles):
    sc.pl.violin(adata, key, groupby="Sample", rotation=90, ax=ax, show=False,
                 stripplot=False)
    ax.set_title(f"{title} — BEFORE filtering", fontsize=10, fontweight="bold")
    ax.set_ylabel(key)

plt.suptitle("QC Metrics BEFORE Filtering", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


### 3b — Apply Filters

In [ ]:
# --- Cell/spot filter --------------------------------------------------------
sc.pp.filter_cells(adata, min_genes=200)

# --- Gene QC parameters ------------------------------------------------------
MIN_SPOTS            = 3   # spots per sample for a gene to "count"
MIN_SAMPLES_IN_GROUP = 3   # samples within a group that must pass

# --- Gene filter: ≥ MIN_SPOTS spots in ≥ MIN_SAMPLES_IN_GROUP samples -------
#     within AT LEAST ONE group (YC / YAD / AC / AAD)
# ---------------------------------------------------------------------------
GROUP_KEY  = "Group"
SAMPLE_KEY = "Sample"

groups  = adata.obs[GROUP_KEY].unique()
n_genes = adata.n_vars

gene_passes_any_group = np.zeros(n_genes, dtype=bool)

print(f"\n  Gene filter: ≥{MIN_SPOTS} spots in ≥{MIN_SAMPLES_IN_GROUP} samples "
      f"within at least 1 of {len(groups)} groups")

for group in sorted(groups):
    group_obs   = adata.obs[GROUP_KEY] == group
    group_adata = adata[group_obs]
    samples     = group_adata.obs[SAMPLE_KEY].unique()

    samples_passing = np.zeros(n_genes, dtype=int)
    for sample in samples:
        sample_obs   = group_adata.obs[SAMPLE_KEY] == sample
        sample_adata = group_adata[sample_obs]
        spot_counts  = np.asarray((sample_adata.X > 0).sum(axis=0)).ravel()
        samples_passing += (spot_counts >= MIN_SPOTS).astype(int)

    group_pass = samples_passing >= MIN_SAMPLES_IN_GROUP
    gene_passes_any_group |= group_pass

    print(f"    {group:>5}: {group_pass.sum():>6,} genes pass "
          f"(≥{MIN_SPOTS} spots in ≥{MIN_SAMPLES_IN_GROUP}/{len(samples)} samples)")

n_before = adata.n_vars
adata    = adata[:, gene_passes_any_group].copy()
n_after  = adata.n_vars

print(f"\n  Genes before filter : {n_before:,}")
print(f"  Genes retained      : {n_after:,}  "
      f"({100 * n_after / n_before:.1f}% of pre-filter genes)")
print(f"  Genes removed       : {n_before - n_after:,}")

# --- Recalculate QC metrics after gene filter --------------------------------
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True)

print(f"\nAfter QC: {adata.shape}")


### 3c — QC Metrics AFTER Filtering

In [ ]:
# --- Violin plots AFTER filtering --------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()

for ax, key, title in zip(axes, qc_keys, qc_titles):
    sc.pl.violin(adata, key, groupby="Sample", rotation=90, ax=ax, show=False,
                 stripplot=False)
    ax.set_title(f"{title} — AFTER filtering", fontsize=10, fontweight="bold")
    ax.set_ylabel(key)

plt.suptitle("QC Metrics AFTER Filtering", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 4 · Normalization

In [ ]:
adata.layers["raw_counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.layers["log_norm"] = adata.X.copy()
adata.raw = adata.copy()

print("Normalization complete (CPM 1e4 + log1p).")
print(f"  Layers: {list(adata.layers.keys())}")

## 5 · Feature Selection (HVGs)

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat_v3",
    batch_key="Sample",
    layer="raw_counts"
)

print(f"HVGs selected: {adata.var['highly_variable'].sum()}")


## 7 · PCA-Loading Gene Filtering


In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=100, use_highly_variable=True, random_state=42)

variance_ratio = adata.uns["pca"]["variance_ratio"]
cumulative_var = np.cumsum(variance_ratio)

sc.pl.pca_variance_ratio(adata, 
                         log=True, 
                         n_pcs=50, 
)

# --- Variance table ----------------------------------------------------------
var_table = pd.DataFrame({
    "PC": range(1, 101),
    "Variance_Ratio": variance_ratio,
    "Log10_Variance_Ratio": np.log10(variance_ratio),
    "Variance_%": variance_ratio * 100,
    "Cumulative_%": cumulative_var * 100,
    "Log10_Cumulative": np.log10(cumulative_var),
})
print("\n" + var_table.to_string(index=False, float_format="%.4f"))


## 8 · Clustering & UMAP & t-SNE

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=30,random_state=42)
sc.tl.umap(adata, random_state=42, min_dist=0.2, spread=5.0)
sc.tl.tsne(adata, n_pcs=30, random_state=42, perplexity=2000)
sc.tl.leiden(adata, resolution=0.5, random_state=42)

In [ ]:
group_order = sorted(adata.obs["Group"].unique().tolist())
adata.obs["Group"] = pd.Categorical(
    adata.obs["Group"], categories=group_order, ordered=True
)

_group_palette = ["#2ecc71", "#e74c3c", "#3498db", "#9b59b6",
                  "#f39c12", "#1abc9c", "#e67e22", "#95a5a6"]
umap_palette = _group_palette[: len(group_order)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
sc.pl.umap(adata, color="Group",  ax=axes[0], show=False,
           palette=umap_palette, title="UMAP — Group")
sc.pl.umap(adata, color="leiden", ax=axes[1], show=False,
           title="UMAP — Leiden Clusters")
sc.pl.tsne(adata, color="Group", ax=axes[2], show=False,
           palette=umap_palette, title="t-SNE — Group")
sc.pl.tsne(adata, color="leiden", ax=axes[3], show=False,
           title="t-SNE — Leiden Clusters")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_order = [
    "YC_1","YC_2","YC_3","YC_4",
    "YAD_1","YAD_2","YAD_3","YAD_4",
    "AC_1","AC_2","AC_3","AC_4",
    "AAD_1","AAD_2","AAD_3","AAD_4",
]

clusters = sorted(adata.obs["leiden"].unique())

# One color per cluster
cmap = plt.cm.get_cmap("tab20", len(clusters))
cluster_colors = {cl: cmap(i) for i, cl in enumerate(clusters)}

fig, axes = plt.subplots(4, 4, figsize=(18, 18))

for ax, sample in zip(axes.flat, sample_order):

    ad = adata[adata.obs["Sample"] == sample]

    colors = [cluster_colors[c] for c in ad.obs["leiden"]]

    ax.scatter(
        ad.obsm["spatial"][:, 0],
        ad.obsm["spatial"][:, 1],
        c=colors,
        s=8,
        linewidth=0,
    )

    ax.set_title(sample, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    ax.invert_yaxis()

# Shared legend
handles = [
    plt.Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        color=cluster_colors[c],
        label=f"Cluster {c}",
        markersize=6,
    )
    for c in clusters
]

fig.legend(
    handles=handles,
    loc="center right",
    bbox_to_anchor=(1.08, 0.5),
    title="Leiden"
)

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30,random_state=42)
sc.tl.umap(adata, random_state=42, min_dist=0.5, spread=1)
sc.tl.tsne(adata, n_pcs=30, random_state=42, perplexity=30,)
sc.tl.leiden(adata, resolution=0.5, random_state=42)

In [ ]:
group_order = sorted(adata.obs["Group"].unique().tolist())
adata.obs["Group"] = pd.Categorical(
    adata.obs["Group"], categories=group_order, ordered=True
)

_group_palette = ["#2ecc71", "#e74c3c", "#3498db", "#9b59b6",
                  "#f39c12", "#1abc9c", "#e67e22", "#95a5a6"]
umap_palette = _group_palette[: len(group_order)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
sc.pl.umap(adata, color="Group",  ax=axes[0], show=False,
           palette=umap_palette, title="UMAP — Group")
sc.pl.umap(adata, color="leiden", ax=axes[1], show=False,
           title="UMAP — Leiden Clusters")
sc.pl.tsne(adata, color="Group", ax=axes[2], show=False,
           palette=umap_palette, title="t-SNE — Group")
sc.pl.tsne(adata, color="leiden", ax=axes[3], show=False,
           title="t-SNE — Leiden Clusters")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_order = [
    "YC_1","YC_2","YC_3","YC_4",
    "YAD_1","YAD_2","YAD_3","YAD_4",
    "AC_1","AC_2","AC_3","AC_4",
    "AAD_1","AAD_2","AAD_3","AAD_4",
]

clusters = sorted(adata.obs["leiden"].unique())

# One color per cluster
cmap = plt.cm.get_cmap("tab20", len(clusters))
cluster_colors = {cl: cmap(i) for i, cl in enumerate(clusters)}

fig, axes = plt.subplots(4, 4, figsize=(18, 18))

for ax, sample in zip(axes.flat, sample_order):

    ad = adata[adata.obs["Sample"] == sample]

    colors = [cluster_colors[c] for c in ad.obs["leiden"]]

    ax.scatter(
        ad.obsm["spatial"][:, 0],
        ad.obsm["spatial"][:, 1],
        c=colors,
        s=8,
        linewidth=0,
    )

    ax.set_title(sample, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    ax.invert_yaxis()

# Shared legend
handles = [
    plt.Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        color=cluster_colors[c],
        label=f"Cluster {c}",
        markersize=6,
    )
    for c in clusters
]

fig.legend(
    handles=handles,
    loc="center right",
    bbox_to_anchor=(1.08, 0.5),
    title="Leiden"
)

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30,random_state=42)
sc.tl.umap(adata, random_state=42, min_dist=0.2, spread=5.0)
sc.tl.tsne(adata, n_pcs=30, random_state=42, perplexity=2000)
sc.tl.leiden(adata, resolution=0.5, random_state=42)

In [ ]:
group_order = sorted(adata.obs["Group"].unique().tolist())
adata.obs["Group"] = pd.Categorical(
    adata.obs["Group"], categories=group_order, ordered=True
)

_group_palette = ["#2ecc71", "#e74c3c", "#3498db", "#9b59b6",
                  "#f39c12", "#1abc9c", "#e67e22", "#95a5a6"]
umap_palette = _group_palette[: len(group_order)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
sc.pl.umap(adata, color="Group",  ax=axes[0], show=False,
           palette=umap_palette, title="UMAP — Group")
sc.pl.umap(adata, color="leiden", ax=axes[1], show=False,
           title="UMAP — Leiden Clusters")
sc.pl.tsne(adata, color="Group", ax=axes[2], show=False,
           palette=umap_palette, title="t-SNE — Group")
sc.pl.tsne(adata, color="leiden", ax=axes[3], show=False,
           title="t-SNE — Leiden Clusters")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_order = [
    "YC_1","YC_2","YC_3","YC_4",
    "YAD_1","YAD_2","YAD_3","YAD_4",
    "AC_1","AC_2","AC_3","AC_4",
    "AAD_1","AAD_2","AAD_3","AAD_4",
]

clusters = sorted(adata.obs["leiden"].unique())

# One color per cluster
cmap = plt.cm.get_cmap("tab20", len(clusters))
cluster_colors = {cl: cmap(i) for i, cl in enumerate(clusters)}

fig, axes = plt.subplots(4, 4, figsize=(18, 18))

for ax, sample in zip(axes.flat, sample_order):

    ad = adata[adata.obs["Sample"] == sample]

    colors = [cluster_colors[c] for c in ad.obs["leiden"]]

    ax.scatter(
        ad.obsm["spatial"][:, 0],
        ad.obsm["spatial"][:, 1],
        c=colors,
        s=8,
        linewidth=0,
    )

    ax.set_title(sample, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    ax.invert_yaxis()

# Shared legend
handles = [
    plt.Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        color=cluster_colors[c],
        label=f"Cluster {c}",
        markersize=6,
    )
    for c in clusters
]

fig.legend(
    handles=handles,
    loc="center right",
    bbox_to_anchor=(1.08, 0.5),
    title="Leiden"
)

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

# Workflow A (main analysis): (My path)
* QC
* 3000 HVGs
* PCA
* Keep the top 1000–1500 genes by PCA contribution
* Pseudobulk Welch test
* FDR
* GSEA
# Workflow B (interpretation):
* For each of the first 10–30 PCs, extract the top positively and negatively loaded genes.
* Run pathway enrichment separately for each PC.
* Relate the PCs to age, disease, and your MSI-derived lipid features.

### PCA filtering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_order = [
    "YC_1","YC_2","YC_3","YC_4",
    "YAD_1","YAD_2","YAD_3","YAD_4",
    "AC_1","AC_2","AC_3","AC_4",
    "AAD_1","AAD_2","AAD_3","AAD_4",
]

colors = {
    "YC": "blue",
    "YAD": "red",
    "AC": "green",
    "AAD": "orange",
}

fig, axes = plt.subplots(17, 3, figsize=(15, 80))
axes = axes.ravel()

for i in range(50):

    pcx = 2 * i
    pcy = 2 * i + 1

    ax = axes[i]

    for sample in sample_order:

        ad = adata[adata.obs["Sample"] == sample]

        group = ad.obs["Group"].iloc[0]

        ax.scatter(
            ad.obsm["X_pca"][:, pcx],
            ad.obsm["X_pca"][:, pcy],
            s=8,
            alpha=0.6,
            color=colors[group],
            label=group if sample.endswith("_1") else None,
        )

    ax.set_xlabel(f"PC{pcx+1}")
    ax.set_ylabel(f"PC{pcy+1}")
    ax.set_title(f"PC{pcx+1} vs PC{pcy+1}")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
loadings = adata.varm["PCs"][:, :30]      # genes × PCs
threshold = 0.05

adata.var["pca_filtered"] = (np.abs(loadings) > threshold).any(axis=1)

print(f"PCA-filtered genes: {adata.var['pca_filtered'].sum()}")

## 9 · Pseudobulk Construction

In [ ]:
import anndata as ad
def build_pseudobulk(adata):
    samples = adata.obs["Sample"].unique()
    pb_data, pb_obs = [], []

    for sample in samples:
        mask = adata.obs["Sample"].values == sample
        if sparse.issparse(adata.layers["raw_counts"]):
            s = np.array(adata.layers["raw_counts"][mask].sum(axis=0)).ravel()
        else:
            s = adata.layers["raw_counts"][mask].sum(axis=0)
        pb_data.append(s)

        row = adata.obs.loc[mask].iloc[0]
        pb_obs.append({
            "Sample": sample,
            "Group": row["Group"],
            "Condition": row["Condition"],
            "Age": row["Age"],
            "n_spots": int(mask.sum()),
        })

    pb = ad.AnnData(
        X=np.array(pb_data, dtype=np.float32),
        obs=pd.DataFrame(pb_obs, index=samples),
        var=adata.var[[]].copy(),
    )
    return pb


pb = build_pseudobulk(adata)
pb_raw = pb.copy()

sc.pp.normalize_total(pb, target_sum=1e6)
sc.pp.log1p(pb)

print(f"Pseudobulk: {pb.shape}")
print(pb.obs[["Group", "Condition", "Age", "n_spots"]])


## 10 · Differential Expression (Welch t-test)

In [ ]:
print(f"Active comparisons ({len(COMPARISONS)}):")
for c in COMPARISONS:
    g1_desc = describe_filter(c["group1_filter"])
    g2_desc = describe_filter(c["group2_filter"])
    print(f"  • {c['name']:20s}  {g1_desc}  vs  {g2_desc}")


def run_de_welch(pb, group1_mask, group2_mask, gene_list=None):
    X1 = pb[group1_mask].X
    X2 = pb[group2_mask].X

    genes = pb.var_names if gene_list is None else gene_list
    gene_idx = [list(pb.var_names).index(g) for g in genes]

    results = []
    for i, gi in enumerate(gene_idx):
        vals1 = X1[:, gi]
        vals2 = X2[:, gi]

        if np.std(vals1) == 0 and np.std(vals2) == 0:
            continue

        tstat, pval = stats.ttest_ind(vals1, vals2, equal_var=False)
        if np.isnan(pval):
            pval = 1.0

        cpm1 = np.expm1(vals1)
        cpm2 = np.expm1(vals2)
        mean1 = np.mean(cpm1)
        mean2 = np.mean(cpm2)
        log2fc = np.log2((mean1 + 1e-9) / (mean2 + 1e-9))
        mean_expr = np.mean(np.concatenate([cpm1, cpm2]))

        results.append({
            "gene": genes[i],
            "log2FC": log2fc,
            "mean_expr": mean_expr,
            "pval": pval,
        })

    df = pd.DataFrame(results)
    if df.empty:
        return df
    df["padj"] = multipletests(df["pval"], method="fdr_bh")[1]
    df["method"] = "welch"
    return df


In [ ]:
pca_genes = adata.var_names[adata.var["pca_filtered"]].tolist()

all_de_results = {}

print(f"PCA-filtered genes for testing: {len(pca_genes)}\n")

for comp in COMPARISONS:
    cname = comp["name"]
    cdesc = comp["desc"]

    mask1_pb = build_mask(pb.obs, comp["group1_filter"])
    mask2_pb = build_mask(pb.obs, comp["group2_filter"])

    n1 = mask1_pb.sum()
    n2 = mask2_pb.sum()

    print(f"{'='*60}")
    print(f"DE: {cdesc}")
    print(f"  Group 1 ({describe_filter(comp['group1_filter'])}): {n1} samples")
    print(f"  Group 2 ({describe_filter(comp['group2_filter'])}): {n2} samples")

    if n1 < 2 or n2 < 2:
        print(f"  ⚠ Skipping — need ≥2 samples per side (got {n1} vs {n2})")
        continue

    df = run_de_welch(pb, mask1_pb, mask2_pb, gene_list=pca_genes)
    all_de_results[cname] = df

    sig_up = df[(df["padj"] < 0.05) & (df["log2FC"] > 0.5)].sort_values("log2FC", ascending=False)
    sig_dn = df[(df["padj"] < 0.05) & (df["log2FC"] < -0.5)].sort_values("log2FC", ascending=True)
    print(f"  Sig. up (padj<0.05, |log2FC|>0.5): {len(sig_up)}")
    print(f"  Sig. down:                          {len(sig_dn)}")


    safe_name = cdesc.replace(" ", "_").replace("/", "-")
    df.to_csv(os.path.join(RESULTS_DIR, f"DE_{cname}_{safe_name}.csv"), index=False)
    print()


## 11 · Volcano Plots

In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,
    "ytick.major.size": 2,
})


def _repel_labels_manual(ax, texts_info, fontsize=5):
    fig = ax.get_figure()
    renderer = fig.canvas.get_renderer()
    placed_bboxes = []
    offsets_to_try = [
        (6, 6), (-6, 6), (6, -6), (-6, -6),
        (10, 0), (-10, 0), (0, 10), (0, -10),
        (12, 8), (-12, 8), (12, -8), (-12, -8),
        (16, 0), (-16, 0), (0, 16), (0, -16),
        (14, 12), (-14, 12), (14, -12), (-14, -12),
        (20, 4), (-20, 4), (20, -4), (-20, -4),
        (8, 16), (-8, 16), (8, -16), (-8, -16),
        (22, 10), (-22, 10), (22, -10), (-22, -10),
    ]
    for info in texts_info:
        best_offset = offsets_to_try[0]
        best_overlap = float("inf")
        for dx, dy in offsets_to_try:
            ann = ax.annotate(
                info["gene"], (info["x"], info["y"]),
                fontsize=fontsize, fontstyle="italic",
                xytext=(dx, dy), textcoords="offset points",
                color="0.1", zorder=4,
            )
            fig.canvas.draw()
            bbox = ann.get_window_extent(renderer)
            ann.remove()
            overlap = 0
            for pb_box in placed_bboxes:
                ix = max(0, min(bbox.x1, pb_box.x1) - max(bbox.x0, pb_box.x0))
                iy = max(0, min(bbox.y1, pb_box.y1) - max(bbox.y0, pb_box.y0))
                overlap += ix * iy
            if overlap < best_overlap:
                best_overlap = overlap
                best_offset = (dx, dy)
            if overlap == 0:
                break
        ann_final = ax.annotate(
            info["gene"], (info["x"], info["y"]),
            fontsize=fontsize, fontstyle="italic",
            xytext=best_offset, textcoords="offset points",
            arrowprops=dict(arrowstyle="-", lw=0.3, color="0.4"),
            color="0.1", zorder=4,
        )
        fig.canvas.draw()
        placed_bboxes.append(ann_final.get_window_extent(renderer))


In [ ]:
for comp in COMPARISONS:
    cname = comp["name"]
    if cname not in all_de_results:
        continue
    de_df = all_de_results[cname].copy()
    comp_desc = comp["desc"]

    de_df["neg_log10_padj"] = -np.log10(de_df["padj"].clip(lower=1e-300))

    m_up = (de_df["padj"] < 0.05) & (de_df["log2FC"] > 0.5)
    m_dn = (de_df["padj"] < 0.05) & (de_df["log2FC"] < -0.5)
    m_ns = ~(m_up | m_dn)

    n_up = m_up.sum()
    n_dn = m_dn.sum()
    n_sig_total = n_up + n_dn

    fig_w = max(7.5, 7.5 + n_sig_total * 0.04)
    fig_h = max(6.0, 6.0 + n_sig_total * 0.03)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    ax.scatter(de_df.loc[m_ns, "log2FC"], de_df.loc[m_ns, "neg_log10_padj"],
               s=6, c="#bdc3c7", alpha=0.4, linewidths=0, rasterized=True,
               label="Not significant", zorder=2)
    ax.scatter(de_df.loc[m_dn, "log2FC"], de_df.loc[m_dn, "neg_log10_padj"],
               s=10, c="#2980b9", alpha=0.7, linewidths=0, rasterized=True,
               label="Downregulated", zorder=3)
    ax.scatter(de_df.loc[m_up, "log2FC"], de_df.loc[m_up, "neg_log10_padj"],
               s=10, c="#e74c3c", alpha=0.7, linewidths=0, rasterized=True,
               label="Upregulated", zorder=3)

    ax.axvline(0.5,  color="0.3", ls="--", lw=0.6, zorder=5)
    ax.axvline(-0.5, color="0.3", ls="--", lw=0.6, zorder=5)

    for p_thr in [0.05, 0.01, 0.001]:
        ax.axhline(-np.log10(p_thr), color="0.3", ls="--", lw=0.6, zorder=5)

    ax.set_xlabel("log$_2$ fold change", fontsize=8)
    ax.set_ylabel("$-$log$_{10}$(p$_{adj}$)", fontsize=8)
    ax.set_title(f"Volcano Plot — {comp_desc}", fontsize=9, fontweight="bold")

    ax.autoscale_view()
    x_lo, x_hi = ax.get_xlim()
    for p_thr in [0.05, 0.01, 0.001]:
        y_val = -np.log10(p_thr)
        ax.text(x_hi * 1.01, y_val, f"p={p_thr}",
                fontsize=5, va="center", ha="left", color="0.45",
                clip_on=False, zorder=6)

    # Annotate significant genes
    sig_genes_df = de_df.loc[m_up | m_dn].copy()
    if len(sig_genes_df) > 0:
        texts_info = []
        for _, row in sig_genes_df.iterrows():
            texts_info.append({
                "gene": row["gene"],
                "x": row["log2FC"],
                "y": row["neg_log10_padj"],
                "color": "#e74c3c" if row["log2FC"] > 0 else "#2980b9",
            })

        if _HAS_ADJUSTTEXT:
            texts = []
            for info in texts_info:
                t = ax.text(info["x"], info["y"], info["gene"],
                            fontsize=5, fontstyle="italic", color="0.1", zorder=4)
                texts.append(t)
            adjust_text(
                texts, ax=ax,
                arrowprops=dict(arrowstyle="-", color="0.4", lw=0.3),
                expand_points=(1.5, 1.5), expand_text=(1.2, 1.2),
                force_points=(0.8, 0.8), force_text=(0.5, 0.5), lim=500,
            )
        else:
            texts_info.sort(key=lambda d: -d["y"])
            _repel_labels_manual(ax, texts_info, fontsize=5)

    ax.legend(frameon=True, fancybox=True, framealpha=0.9, edgecolor="0.7",
              loc="upper left", bbox_to_anchor=(1.02, 0.78),
              borderaxespad=0, fontsize=6)

    fig.text(0.98, 0.95, f"▲ Up: {n_up}",
             fontsize=8, fontweight="bold", color="#e74c3c",
             ha="right", va="top", transform=fig.transFigure)
    fig.text(0.98, 0.91, f"▼ Down: {n_dn}",
             fontsize=8, fontweight="bold", color="#2980b9",
             ha="right", va="top", transform=fig.transFigure)

    plt.tight_layout(rect=[0, 0, 0.82, 1.0])

    vol_path = os.path.join(FIGURE_DIR, f"Volcano_{cname}.png")
    fig.savefig(vol_path, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(vol_path.replace(".png", ".pdf"), bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  Saved: {vol_path}  ({n_sig_total} sig. genes labeled)")


## 12 · Summary

In [ ]:
print("=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)
print(f"QC filter:                  ≥{MIN_SPOTS} spots in ≥{MIN_SAMPLES_IN_GROUP}/4 samples per group")
print(f"DE method:                  Welch t-test")
print(f"Total spots after QC:       {adata.n_obs:,}")
print(f"Total genes after QC:       {adata.n_vars:,}")
print(f"HVGs (Seurat v3):           {adata.var['highly_variable'].sum():,}")
print(f"PCA-filtered genes:         {adata.var['pca_filtered'].sum():,}")
print(f"Leiden clusters:            {adata.obs['leiden'].nunique()}")
print()

print(f"Comparisons run: {len(all_de_results)}")
for comp in COMPARISONS:
    cname = comp["name"]
    if cname not in all_de_results:
        print(f"  ✗ {comp['desc']} — SKIPPED")
        continue
    de = all_de_results[cname]
    n_up = de[(de["padj"] < 0.05) & (de["log2FC"] > 0.5)].shape[0]
    n_dn = de[(de["padj"] < 0.05) & (de["log2FC"] < -0.5)].shape[0]
    print(f"  ✓ {comp['desc']}:")
    print(f"      {describe_filter(comp['group1_filter'])}  vs  {describe_filter(comp['group2_filter'])}")
    print(f"      ▲ Up: {n_up}   ▼ Down: {n_dn}")
print()

print(f"Results saved to: {RESULTS_DIR}/")
print(f"Figures saved to: {FIGURE_DIR}/")
print("Pipeline complete.")
